# 02 - Tratamento do BACEN, JOIN e Regressão Logística
**Projeto:** Confiança, transparência da comunicação e churn em crises bancárias (Etapa 2)

**Ordem do notebook:**
1. Ler e tratar a tabela do BACEN (T2/2026): uma linha por banco
2. Calcular `tx_reclamacao_geral_bacen` e `taxa_procedencia_bacen`
3. Padronizar os nomes e fazer o JOIN com a base limpa do Forms
4. Conferir quem ficou sem par
5. Rodar o Logit simples e calcular as Odds Ratios
6. Salvar `data/processed/dataset_final_logit.csv`

> Rode antes o notebook 01, que gera `dados_limpos_anti_gigo.csv`.

In [1]:
import numpy as np
import pandas as pd

BACEN_PATH = "../data/raw/bacen_t2_2026.csv"
FORMS_PATH = "../data/processed/dados_limpos_anti_gigo.csv"
OUT_PATH   = "../data/processed/dataset_final_logit.csv"


## 1. Ler a tabela do BACEN
O arquivo do BACEN tem três particularidades: separador `;`, número com vírgula decimal e codificação `latin-1` (por isso o `encoding`). Usamos as colunas **por posição**, porque alguns nomes têm caracteres estranhos.

In [2]:
bacen_raw = pd.read_csv(BACEN_PATH, sep=";", encoding="latin-1", decimal=",")
bacen_raw = bacen_raw.iloc[:, :14]          # a 15ª coluna é vazia

bacen = bacen_raw.copy()
bacen.columns = [
    "ano", "trimestre", "categoria", "tipo", "cnpj", "instituicao",
    "indice_bacen", "reclamacoes_respondidas", "reclamacoes_procedentes",
    "reclamacoes_procedentes_extrapoladas", "reclamacoes_analisadas",
    "clientes_ccs_scr", "clientes_ccs", "clientes_scr",
]
print("Linhas:", len(bacen), "| Trimestre:", bacen["trimestre"].unique(), "| Ano:", bacen["ano"].unique())
print("CNPJ vazio em", round((bacen["cnpj"].fillna("").str.strip() == "").mean() * 100, 1), "% das linhas")
bacen.head(3)

Linhas: 198 | Trimestre: <StringArray>
['2º']
Length: 1, dtype: str | Ano: [2026]
CNPJ vazio em 96.0 % das linhas


,ano,trimestre,categoria,tipo,cnpj,instituicao,indice_bacen,reclamacoes_respondidas,reclamacoes_procedentes,reclamacoes_procedentes_extrapoladas,reclamacoes_analisadas,clientes_ccs_scr,clientes_ccs,clientes_scr
0,2026,2º,"Top 15 - Bancos, Financeiras e Instituições de...",Conglomerado,,99PAY IP (conglomerado),"6,01",1448,68,170,580,28239602,28239599,3
1,2026,2º,"Demais Bancos, Financeiras e Instituições de P...",Conglomerado,,ABC-BRASIL (conglomerado),,30,3,5,18,46972,42660,8644
2,2026,2º,"Demais Bancos, Financeiras e Instituições de P...",Conglomerado,,ADYEN DO BRASIL IP (conglomerado),,26,8,9,23,1898,1786,114


## 2. Calcular as taxas do BACEN
- **`tx_reclamacao_geral_bacen` (Z3):** reclamações procedentes extrapoladas por **milhão de clientes**. É a mesma conta do "Índice" oficial, mas o índice vem vazio em muitos bancos, então recalculamos para todos.
- **`taxa_procedencia_bacen` (Z4):** % das reclamações analisadas que foram procedentes (`procedentes / analisadas`).

> **Confirme com o grupo** se a definição de Z4 é essa. A conta é uma escolha nossa, a partir das colunas da tabela.

In [3]:
bacen["tx_reclamacao_geral_bacen"] = (
    bacen["reclamacoes_procedentes_extrapoladas"] / bacen["clientes_ccs_scr"] * 1_000_000
)
bacen["taxa_procedencia_bacen"] = (
    bacen["reclamacoes_procedentes"] / bacen["reclamacoes_analisadas"] * 100
)

# Conferência: nosso cálculo deve bater com o Índice oficial nos bancos do "Top 15"
# (nas instituições pequenas o BACEN usa outro método e há pequenas diferenças)
oficial = pd.to_numeric(bacen["indice_bacen"].astype(str).str.replace(",", "."), errors="coerce")
top15 = bacen["categoria"].str.startswith("Top 15")
dif = (bacen.loc[top15, "tx_reclamacao_geral_bacen"] - oficial[top15]).abs().dropna()
print("Bancos do Top 15 conferidos:", len(dif), "| maior diferença:", round(dif.max(), 3))

Bancos do Top 15 conferidos: 15 | maior diferença: 0.023


## 3. Padronizar os nomes e criar a chave do JOIN
A chave é o nome padronizado do banco (`instituicao_principal`), igual ao do notebook 01.
O dicionário abaixo liga o **nome no Forms** ao **nome no BACEN**. Se algum banco novo aparecer nas respostas, é só incluir uma linha.

In [4]:
MAPA_FORMS_PARA_BACEN = {
    "Nubank": "NU PAGAMENTOS (conglomerado)",
    "Inter":  "INTER (conglomerado)",
    "Itaú":   "ITAU (conglomerado)",
    "Caixa":  "CAIXA ECONÔMICA FEDERAL (conglomerado)",
    # "Outros" não tem equivalente: ficará sem dados do BACEN
}

# Confere se todos os nomes do dicionário existem na tabela
faltando = [v for v in MAPA_FORMS_PARA_BACEN.values() if v not in set(bacen["instituicao"])]
assert not faltando, f"Nomes não encontrados no BACEN: {faltando}"

bacen_por_banco = (
    bacen[bacen["instituicao"].isin(MAPA_FORMS_PARA_BACEN.values())]
    .assign(instituicao_principal=lambda d: d["instituicao"].map({v: k for k, v in MAPA_FORMS_PARA_BACEN.items()}))
    [["instituicao_principal", "tx_reclamacao_geral_bacen", "taxa_procedencia_bacen"]]
    .reset_index(drop=True)
)
bacen_por_banco.round(2)

,instituicao_principal,tx_reclamacao_geral_bacen,taxa_procedencia_bacen
0,Caixa,23.32,16.14
1,Inter,26.16,15.34
2,Itaú,46.87,15.15
3,Nubank,9.21,4.76


## 4. JOIN com o Forms e conferência
Usamos `how="left"`: **nenhuma resposta do Forms é perdida**. Depois conferimos quem ficou sem dados do BACEN.

In [5]:
forms = pd.read_csv(FORMS_PATH)
print("Forms limpo:", forms.shape)

df = forms.merge(bacen_por_banco, on="instituicao_principal", how="left", validate="m:1")
print("Após JOIN:", df.shape, "(deve ter o mesmo número de linhas do Forms)")

sem_par = df[df["tx_reclamacao_geral_bacen"].isna()]
print()
print("Respostas SEM dados do BACEN:", len(sem_par))
print(sem_par["instituicao_principal"].value_counts().to_string())

Forms limpo: (21, 9)
Após JOIN: (21, 11) (deve ter o mesmo número de linhas do Forms)

Respostas SEM dados do BACEN: 2
instituicao_principal
Outros    2


**Decisão para o modelo:** as respostas de "Outros" não têm Z3 e Z4. Para o modelo, usamos só as linhas com dados completos e registramos o descarte (vai para o Diário de Bordo e para a Seção 1.2).

In [6]:
df_modelo = df.dropna(subset=["tx_reclamacao_geral_bacen", "taxa_procedencia_bacen"]).copy()
print("Linhas para o modelo:", len(df_modelo), "de", len(df))
print("Casos de churn:", int(df_modelo["flag_risco_churn"].sum()))
df_modelo.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("Salvo em:", OUT_PATH)

Linhas para o modelo: 19 de 21
Casos de churn: 5
Salvo em: ../data/processed/dataset_final_logit.csv


## 5. Regressão Logística (modelo simples)
**Atenção:** com poucos casos de churn, este modelo é **demonstrativo**. Ele mostra o pipeline funcionando, mas não permite conclusões. Regra prática: cerca de 10 eventos por variável explicativa.

Modelo: `flag_risco_churn ~ percepcao_transparencia`

In [7]:
import statsmodels.formula.api as smf

modelo = smf.logit("flag_risco_churn ~ percepcao_transparencia", data=df_modelo).fit(disp=False)
print(modelo.summary())

                           Logit Regression Results                           
Dep. Variable:       flag_risco_churn   No. Observations:                   19
Model:                          Logit   Df Residuals:                       17
Method:                           MLE   Df Model:                            1
Date:                Wed, 11 Nov 2026   Pseudo R-squ.:                 0.02537
Time:                        02:31:02   Log-Likelihood:                -10.673
converged:                       True   LL-Null:                       -10.950
Covariance Type:            nonrobust   LLR p-value:                    0.4560
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   0.2946      1.845      0.160      0.873      -3.322       3.912
percepcao_transparencia    -0.4454      0.610     -0.730      0.465      -1.641       0.750


### Tabela de resultados (Seção 3.2 do relatório)
O **Odds Ratio** é `e^β`: quanto a chance de churn é multiplicada a cada 1 ponto a mais de transparência. Abaixo de 1 = efeito protetor. Acima de 1 = a chance aumenta.

In [8]:
ic = modelo.conf_int()
tabela = pd.DataFrame({
    "Coeficiente (β)": modelo.params,
    "Erro padrão": modelo.bse,
    "p-valor": modelo.pvalues,
    "Odds Ratio (e^β)": np.exp(modelo.params),
    "IC 95% inf.": np.exp(ic[0]),
    "IC 95% sup.": np.exp(ic[1]),
})
tabela.loc["Intercept", "Odds Ratio (e^β)"] = np.nan   # OR do intercepto não é interpretado
tabela.round(3)

,Coeficiente (β),Erro padrão,p-valor,Odds Ratio (e^β),IC 95% inf.,IC 95% sup.
Intercept,0.295,1.845,0.873,NaN,0.036,49.979
percepcao_transparencia,-0.445,0.610,0.465,0.641,0.194,2.117


In [9]:
print("Pseudo R² de McFadden:", round(modelo.prsquared, 4))
print("Log-Likelihood:       ", round(modelo.llf, 3))
print("P-valor do teste LR:  ", round(modelo.llr_pvalue, 4))
print("N usado no modelo:    ", int(modelo.nobs))

Pseudo R² de McFadden: 0.0254
Log-Likelihood:        -10.673
P-valor do teste LR:   0.456
N usado no modelo:     19


### Como ler o resultado (para a apresentação)
- Olhe o **p-valor** de `percepcao_transparencia`. Com poucos casos, é comum ele ser alto (> 0,05). Isso quer dizer que os dados **não permitem afirmar** que existe efeito, não que o efeito não exista.
- Olhe o **intervalo de confiança** do Odds Ratio. Se ele for muito largo e incluir o 1, o resultado é inconclusivo.
- Registre no Diário de Bordo: *"Com N pequeno, o Logit foi usado para validar o pipeline; o modelo completo depende de ampliar a coleta."*

**Próximos passos quando o N crescer** (em ordem): incluir `tempo_relacionamento_anos` e `faixa_renda_perfil`; depois o canal (`C(canal_comunicacao_pref)`); por último `tx_reclamacao_geral_bacen` e `taxa_procedencia_bacen`, usando erros-padrão agrupados por banco (`cov_type="cluster"`).